In [9]:
import pandas as pd

train_df = pd.read_csv(
    "../data/processed/train_transactions.csv",
    parse_dates=["InvoiceDate"]
)

train_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [10]:
# Remove missing customers
train_df = train_df.dropna(subset=["CustomerID"]).copy()

# Create revenue
train_df["Revenue"] = train_df["Quantity"] * train_df["UnitPrice"]

# Remove returns / negative values
train_df = train_df[(train_df["Quantity"] > 0) & (train_df["Revenue"] > 0)].copy()

train_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [4]:
snapshot_date = train_df["InvoiceDate"].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2011-09-10 15:53:00')

In [5]:
preference_df = train_df.groupby(["CustomerID", "Description"]).agg(
    frequency=("InvoiceNo", "nunique"),
    monetary=("Revenue", "sum"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

# Recency
preference_df["recency_days"] = (
    snapshot_date - preference_df["last_purchase"]
).dt.days

preference_df.head()

,CustomerID,Description,frequency,monetary,last_purchase,recency_days
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.6,2011-01-18 10:01:00,235
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.5,2011-08-02 08:48:00,39
2,12347.0,3D SHEET OF CAT STICKERS,1,10.2,2011-04-07 10:43:00,156
3,12347.0,3D SHEET OF DOG STICKERS,1,10.2,2011-04-07 10:43:00,156
4,12347.0,60 TEATIME FAIRY CAKE CASES,2,26.4,2011-08-02 08:48:00,39


In [6]:
preference_df = preference_df.drop(columns=["last_purchase"])

In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

preference_df[["freq_norm", "monetary_norm", "recency_norm"]] = scaler.fit_transform(
    preference_df[["frequency", "monetary", "recency_days"]]
)

In [11]:
preference_df["recency_norm"] = 1 - preference_df["recency_norm"]

In [12]:
preference_df["preference_score"] = (
    0.5 * preference_df["freq_norm"] +
    0.3 * preference_df["monetary_norm"] +
    0.2 * preference_df["recency_norm"]
)

preference_df.head()

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.6,235,0.000000,1.000000,0.170213,0.334043
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.5,39,0.062500,0.003440,0.865248,0.205332
2,12347.0,3D SHEET OF CAT STICKERS,1,10.2,156,0.000000,0.000132,0.450355,0.090111
3,12347.0,3D SHEET OF DOG STICKERS,1,10.2,156,0.000000,0.000132,0.450355,0.090111
4,12347.0,60 TEATIME FAIRY CAKE CASES,2,26.4,39,0.020833,0.000342,0.865248,0.183569


In [13]:
preference_df = preference_df.sort_values(
    ["CustomerID", "preference_score"],
    ascending=[True, False]
)

In [14]:
top_n = 5

top_preferences = preference_df.groupby("CustomerID").head(top_n)

top_preferences.head(20)

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.60,235,0.000000,1.000000,0.170213,0.334043
6,12347.0,AIRLINE BAG VINTAGE JET SET BROWN,5,85.00,39,0.083333,0.001101,0.865248,0.215047
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.50,39,0.062500,0.003440,0.865248,0.205332
15,12347.0,ALARM CLOCK BAKELIKE RED,4,90.00,39,0.062500,0.001166,0.865248,0.204649
55,12347.0,REGENCY CAKESTAND 3 TIER,3,114.75,39,0.041667,0.001487,0.865248,0.194329
9,12347.0,AIRLINE BAG VINTAGE TOKYO 78,3,85.00,39,0.041667,0.001101,0.865248,0.194213
100,12348.0,POSTAGE,3,320.00,158,0.041667,0.004146,0.443262,0.110730
88,12348.0,ICE CREAM PEN LIP GLOSS,1,120.00,158,0.000000,0.001555,0.443262,0.089119
87,12348.0,DOUGHNUT LIP GLOSS,1,100.00,158,0.000000,0.001296,0.443262,0.089041
89,12348.0,ICE CREAM SUNDAE LIP GLOSS,1,90.00,158,0.000000,0.001166,0.443262,0.089002


In [15]:
customer_id = 12347

top_preferences[top_preferences["CustomerID"] == customer_id]

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score
6,12347.0,AIRLINE BAG VINTAGE JET SET BROWN,5,85.00,39,0.083333,0.001101,0.865248,0.215047
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.50,39,0.062500,0.003440,0.865248,0.205332
15,12347.0,ALARM CLOCK BAKELIKE RED,4,90.00,39,0.062500,0.001166,0.865248,0.204649
55,12347.0,REGENCY CAKESTAND 3 TIER,3,114.75,39,0.041667,0.001487,0.865248,0.194329
9,12347.0,AIRLINE BAG VINTAGE TOKYO 78,3,85.00,39,0.041667,0.001101,0.865248,0.194213


## Moving from Heuristic Weights to Data-Driven Weights

At this stage, we successfully built an initial preference modeling pipeline using a simple weighted scoring formula based on:

- Frequency
- Monetary value
- Recency

This first version was useful as a practical starting point because it allowed us to generate ranked product preferences for each customer in a simple and interpretable way.

However, the weights used in the score formula (for example, `0.5`, `0.3`, and `0.2`) were chosen manually based on intuition rather than learned from the data itself. While this is acceptable for an initial prototype, it introduces an important limitation: the ranking depends on subjective assumptions about which feature is more important.

To make the preference model more robust and evidence-based, we decided to move to a more advanced approach where the weights are derived from the data rather than assigned manually.

The motivation for this transition is:

- Manual weights are heuristic and may not reflect real customer behavior.
- Different datasets may require different importance levels for frequency, monetary value, and recency.
- A data-driven approach can produce weights that are more defensible and better aligned with actual business outcomes.

Therefore, instead of continuing with fixed heuristic weights, the next step is to estimate the feature importance using a simple predictive model and use the learned coefficients as more reliable preference weights.

This transition improves the preference modeling component by making it:

- more principled,
- more adaptive to the dataset,
- and more professional from a business and machine learning perspective.

In [19]:
# load revenue dataset
revenue_df = pd.read_csv("../data/processed/revenue_dataset.csv")

# merge
merged = preference_df.merge(
    revenue_df[["CustomerID", "future_revenue"]],
    on="CustomerID",
    how="inner"
)

merged.head()

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score,future_revenue
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.60,235,0.000000,1.000000,0.170213,0.334043,0.00
1,12347.0,AIRLINE BAG VINTAGE JET SET BROWN,5,85.00,39,0.083333,0.001101,0.865248,0.215047,1519.14
2,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.50,39,0.062500,0.003440,0.865248,0.205332,1519.14
3,12347.0,ALARM CLOCK BAKELIKE RED,4,90.00,39,0.062500,0.001166,0.865248,0.204649,1519.14
4,12347.0,REGENCY CAKESTAND 3 TIER,3,114.75,39,0.041667,0.001487,0.865248,0.194329,1519.14


## Why Use Future Revenue to Learn Weights?

To move from manual weights to a data-driven approach, we need a signal that reflects true customer value.

We use **future revenue** because:

- It represents how valuable the customer will be.
- It captures real behavior (spending + engagement).
- It aligns directly with business goals.

Instead of assuming which feature is more important, we train a simple model:

future_revenue = w1 * frequency + w2 * monetary + w3 * recency

The learned coefficients (`w1`, `w2`, `w3`) are then used as preference weights.

This makes the scoring:

- data-driven instead of heuristic,
- more reliable,
- and better aligned with real customer behavior.

In [20]:
from sklearn.linear_model import LinearRegression

# Features
X = merged[["freq_norm", "monetary_norm", "recency_norm"]]

# Target
y = merged["future_revenue"]

# Train model
model = LinearRegression()
model.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [21]:
weights = model.coef_

print("Frequency weight:", weights[0])
print("Monetary weight:", weights[1])
print("Recency weight:", weights[2])

Frequency weight: 62241.01509956132
Monetary weight: 309749.7257973836
Recency weight: 634.3585185626871


## Interpreting Learned Weights and Final Decision

After training a regression model to learn weights using **future revenue** as the target, we observed that the resulting weights are heavily influenced by monetary value.

This outcome is expected because the model is optimized to answer:

> "Which features are most related to future spending?"

As a result:
- Monetary value received the highest importance.
- Frequency and recency had a much smaller impact.

However, this creates a mismatch with our actual goal.

### Key Observation

- The learned weights reflect **revenue importance**, not necessarily **customer preference**.
- In preference modeling, repeated behavior (frequency) is often a stronger signal of true interest than one-time high spending.

### Decision

To address this mismatch, we do not rely solely on model-learned weights.

Instead, we adopt a **hybrid approach**:
- Combine **data-driven weights** (from the model)
- With **manually defined weights** (based on business intuition)

This allows us to balance:
- real data signals (revenue behavior),
- and logical assumptions about customer preferences.

As a result, the final weights are more aligned with both:
- business understanding,
- and observed data patterns.

In [29]:
# normalize model weights
w_model = np.abs(w_model)
w_model = w_model / w_model.sum()

# manual already normalized
w_manual = np.array([0.6, 0.25, 0.15])

# mix
final_weights = 0.3 * w_model + 0.7 * w_manual

print(final_weights)

[0.47011016 0.42437912 0.10551072]


In [31]:
w1, w2, w3 = final_weights

preference_df["preference_score"] = (
    w1 * preference_df["freq_norm"] +
    w2 * preference_df["monetary_norm"] +
    w3 * preference_df["recency_norm"]
)

preference_df.head(20)

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.60,235,0.000000,1.000000,0.170213,0.442338
6,12347.0,AIRLINE BAG VINTAGE JET SET BROWN,5,85.00,39,0.083333,0.001101,0.865248,0.130936
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.50,39,0.062500,0.003440,0.865248,0.122135
15,12347.0,ALARM CLOCK BAKELIKE RED,4,90.00,39,0.062500,0.001166,0.865248,0.121170
55,12347.0,REGENCY CAKESTAND 3 TIER,3,114.75,39,0.041667,0.001487,0.865248,0.111512
9,12347.0,AIRLINE BAG VINTAGE TOKYO 78,3,85.00,39,0.041667,0.001101,0.865248,0.111348
8,12347.0,AIRLINE BAG VINTAGE JET SET WHITE,3,51.00,39,0.041667,0.000661,0.865248,0.111161
79,12347.0,VINTAGE HEADS AND TAILS CARD GAME,2,45.00,39,0.020833,0.000583,0.865248,0.101334
7,12347.0,AIRLINE BAG VINTAGE JET SET RED,2,34.00,39,0.020833,0.000440,0.865248,0.101274
4,12347.0,60 TEATIME FAIRY CAKE CASES,2,26.40,39,0.020833,0.000342,0.865248,0.101232


### Issues with Initial Preference Scoring

The initial preference scoring approach produced unrealistic results.  
Some products with very high monetary value (one-time expensive purchases) were ranked as top preferences, even when the customer only bought them once.

This behavior does not reflect true customer preference, because:
- Preference should prioritize **repeated behavior (frequency)**  
- Not just **high spending in a single transaction**

### Why We Improved the Approach

To fix this, we introduced **customer-level normalization** and adjusted the weighting logic:

- Normalize features per customer → avoid dominance of outliers  
- Reduce the impact of extreme monetary values  
- Emphasize frequency as the main signal of preference  

This results in a more realistic representation of what customers actually prefer.

In [32]:
preference_df["monetary_log"] = np.log1p(preference_df["monetary"])

In [33]:
preference_df["freq_norm"] = preference_df.groupby("CustomerID")["frequency"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

In [34]:
preference_df["monetary_norm"] = preference_df.groupby("CustomerID")["monetary_log"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

In [35]:
preference_df["recency_norm"] = preference_df.groupby("CustomerID")["recency_days"].transform(
    lambda x: (x.max() - x) / (x.max() - x.min() + 1e-9)
)

In [36]:
w1, w2, w3 = final_weights

preference_df["preference_score"] = (
    w1 * preference_df["freq_norm"] +
    w2 * preference_df["monetary_norm"] +
    w3 * preference_df["recency_norm"]
)

In [37]:
preference_df.sort_values(["CustomerID", "preference_score"], ascending=[True, False])

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score,monetary_log
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.60,235,0.00,0.000000,0.0,0.000000,11.253955
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.50,39,0.75,1.000000,1.0,0.882472,5.585374
6,12347.0,AIRLINE BAG VINTAGE JET SET BROWN,5,85.00,39,1.00,0.701337,1.0,0.873254,4.454347
15,12347.0,ALARM CLOCK BAKELIKE RED,4,90.00,39,0.75,0.716260,1.0,0.762059,4.510860
55,12347.0,REGENCY CAKESTAND 3 TIER,3,114.75,39,0.50,0.779787,1.0,0.671491,4.751433
...,...,...,...,...,...,...,...,...,...,...
168971,18287.0,STRAWBERRY CERAMIC TRINKET BOX,1,15.00,111,0.00,0.161506,0.0,0.068540,2.772589
168967,18287.0,SMALL PURPLE BABUSHKA NOTEBOOK,1,10.20,111,0.00,0.004735,0.0,0.002009,2.415914
168968,18287.0,SMALL RED BABUSHKA NOTEBOOK,1,10.20,111,0.00,0.004735,0.0,0.002009,2.415914
168969,18287.0,SMALL YELLOW BABUSHKA NOTEBOOK,1,10.20,111,0.00,0.004735,0.0,0.002009,2.415914


In [38]:
top_products = (
    preference_df
    .sort_values(["CustomerID", "preference_score"], ascending=[True, False])
    .groupby("CustomerID")
    .head(5)
)

top_products.head(20)

,CustomerID,Description,frequency,monetary,recency_days,freq_norm,monetary_norm,recency_norm,preference_score,monetary_log
0,12346.0,MEDIUM CERAMIC TOP STORAGE JAR,1,77183.60,235,0.00,0.000000,0.000000,0.000000,11.253955
1,12347.0,3D DOG PICTURE PLAYING CARDS,4,265.50,39,0.75,1.000000,1.000000,0.882472,5.585374
6,12347.0,AIRLINE BAG VINTAGE JET SET BROWN,5,85.00,39,1.00,0.701337,1.000000,0.873254,4.454347
15,12347.0,ALARM CLOCK BAKELIKE RED,4,90.00,39,0.75,0.716260,1.000000,0.762059,4.510860
55,12347.0,REGENCY CAKESTAND 3 TIER,3,114.75,39,0.50,0.779787,1.000000,0.671491,4.751433
9,12347.0,AIRLINE BAG VINTAGE TOKYO 78,3,85.00,39,0.50,0.701337,1.000000,0.638199,4.454347
100,12348.0,POSTAGE,3,320.00,158,1.00,1.000000,1.000000,1.000000,5.771441
88,12348.0,ICE CREAM PEN LIP GLOSS,1,120.00,158,0.00,0.661358,1.000000,0.386177,4.795791
87,12348.0,DOUGHNUT LIP GLOSS,1,100.00,158,0.00,0.598649,1.000000,0.359565,4.615121
89,12348.0,ICE CREAM SUNDAE LIP GLOSS,1,90.00,158,0.00,0.562461,1.000000,0.344207,4.510860
